# Lecture 3 — Big Geo Data in Python

**R-tree · GeoParquet · Dask-GeoPandas · Anzony Quispe · 2026**

## What you'll learn

- How **spatial indexes (R-tree)** prune candidate pairs and make joins orders of magnitude faster
- How **Apache GeoParquet** replaces slow, size-capped Shapefiles with a columnar, cloud-native format
- How **Dask-GeoPandas** partitions a GeoDataFrame and runs geospatial operations in parallel
- When and why each technique matters as dataset size grows beyond RAM

## 0 · Setup

Install dependencies if needed (uncomment the line below).

In [ ]:
# !pip install geopandas dask-geopandas numpy pandas matplotlib pyarrow pyogrio

In [ ]:
import warnings, os
warnings.filterwarnings("ignore")
os.environ["USE_PYGEOS"] = "0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import pyarrow
import dask_geopandas

print("geopandas     :", gpd.__version__)
print("pyarrow       :", pyarrow.__version__)
print("dask_geopandas:", dask_geopandas.__version__)

## 1 · Download data & build variables

The cells below download the **Natural Earth 110m countries** shapefile and
construct all the variables the lecture examples depend on:
`world`, `south_america`, and `big` (a synthetic dataset of 200 000 random points).

In [ ]:
import urllib.request, zipfile, tempfile

URL = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
tmp = tempfile.mkdtemp()
zip_path = os.path.join(tmp, "ne.zip")
urllib.request.urlretrieve(URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(tmp)

world = gpd.read_file(os.path.join(tmp, "ne_110m_admin_0_countries.shp"))
world = world[["NAME", "CONTINENT", "POP_EST", "geometry"]]

# Also write the round-trip files that Section 13 reads back
world.to_file("/tmp/world.gpkg", driver="GPKG")
world.to_file("/tmp/world.geojson", driver="GeoJSON")

print("world loaded:", len(world), "countries")

south_america = world[world.CONTINENT == "South America"].copy()
print("south_america:", len(south_america), "countries")

In [ ]:
# Synthetic dataset of 200 000 random global points — used in Sections 12 & 14
big = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(
        np.random.uniform(-180, 180, 200_000),
        np.random.uniform(-60,   85, 200_000),
    ),
    crs="EPSG:4326",
)
print("big dataset:", len(big), "points")

## 12 · Spatial indexing (R-tree)

When you need to find which points fall inside which polygons, the naive approach checks every point against every polygon — O(N·M) comparisons that become prohibitively slow at scale.

An **R-tree** (rectangle tree) organises geometries into a hierarchy of bounding boxes. A query first prunes bounding boxes that cannot possibly intersect, reducing comparisons to O(N log M). GeoPandas builds an R-tree index automatically on any GeoDataFrame via `.sindex` — it is the engine behind fast `sjoin` operations on millions of rows.

### 12a · Spatial indexing (R-tree)

Naïve "for every A check every B" is O(N·M).
A **spatial index** (R-tree) prunes pairs whose bounding boxes don't overlap → ~O(N log M).

GeoPandas builds one automatically under the hood (`gdf.sindex`).
You almost never call it directly, but it's why `sjoin` is fast even on millions of rows.

In [ ]:
big = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(
        np.random.uniform(-180, 180, 200_000),
        np.random.uniform(-60,  85,  200_000),
    ),
    crs="EPSG:4326",
)
print("Built an R-tree over", len(big), "points.")
print("Index type:", type(big.sindex).__name__)

# Query: which points fall inside South America's bounding box?
minx, miny, maxx, maxy = south_america.total_bounds
candidates = list(big.sindex.intersection((minx, miny, maxx, maxy)))
print("Candidates after R-tree prune:", len(candidates))

## 13 · Heavy data with **Apache GeoParquet**

As datasets grow, the Shapefile format quickly becomes a bottleneck: it is capped at 2 GB, uses truncated 10-character column names, and reads/writes slowly because it is row-oriented.

**Apache GeoParquet** solves all of these problems. It is the geospatial extension of the Parquet columnar format — the same foundation used by Spark, DuckDB, and Polars. Benefits for big geo work:

- **5–20× smaller** files thanks to columnar compression
- **Predicate pushdown**: read only the columns or row groups you need
- **Cloud-native**: streams from S3, GCS, or Azure Blob without a full download
- **CRS stored as metadata** — no sidecar `.prj` file to lose
- **Arrow-native**: zero-copy interop with DuckDB, Polars, Dask, and pandas

### 13a · Heavy data with **Apache GeoParquet**

Shapefiles are slow, capped at 2 GB, and have 10-char column names.
**GeoParquet** is the modern alternative:

- Columnar, compressed → 5-20× smaller than Shapefile
- Streams from cloud storage (S3, GCS) without a full download
- Stores CRS as metadata
- Native to the Arrow ecosystem (DuckDB, Polars, Dask)

In [ ]:
# Round-trip the world dataset to GeoParquet
world.to_parquet("/tmp/world.parquet")          # writes GeoParquet
back = gpd.read_parquet("/tmp/world.parquet")

import os
print("Shapefile size :", os.path.getsize("/tmp/world.gpkg"),    "bytes (gpkg)")
print("GeoJSON   size :", os.path.getsize("/tmp/world.geojson"), "bytes")
print("Parquet   size :", os.path.getsize("/tmp/world.parquet"), "bytes")
print("CRS preserved? ", back.crs == world.crs)

In [ ]:
# Filter on read — only read columns + rows you need
small = gpd.read_parquet(
    "/tmp/world.parquet",
    columns=["NAME", "CONTINENT", "geometry"],
)
small.head()

## 14 · Parallel processing with **Dask-GeoPandas**

Even GeoParquet and R-trees hit a wall when data does not fit in a single machine's RAM, or when a single-threaded computation is simply too slow.

**Dask-GeoPandas** extends GeoPandas with lazy, parallel execution:

- Splits a GeoDataFrame into **partitions** (spatial or row-based)
- Each partition is processed **in parallel** across CPU cores or a cluster
- Operations are **lazy** — nothing runs until you call `.compute()`
- Exposes the same familiar API as GeoPandas (`.dissolve`, `.sjoin`, `.buffer`, …)
- Reads and writes GeoParquet partitions directly, enabling out-of-core workflows

### 14a · Parallel processing with **Dask-GeoPandas**

When your GeoDataFrame won't fit in RAM (or is just slow):

- **dask-geopandas** partitions the data and runs ops in parallel
- Same API as GeoPandas (`.dissolve`, `.sjoin`, …)
- Reads/writes GeoParquet partitions directly

In [ ]:
import dask_geopandas as dgpd

# Pretend `big` is enormous; split into 8 partitions
dbig = dgpd.from_geopandas(big, npartitions=8)
print(dbig)

# Lazy: nothing has been computed yet
result = dbig.sjoin(south_america[["NAME", "geometry"]], predicate="within")
print("type:", type(result).__name__)

# Trigger computation
points_in_sa = result.compute()
print("Points in South America:", len(points_in_sa))